In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pydeck as pdk
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta

import requests
import time
import re

from IPython.display import display
import ipywidgets as widgets

import geopandas as gpd
from dash import Dash, dcc, html, Input, Output



## node county map

In [3]:
node_cty_map = pd.read_csv('data/pnode_county_mapping.csv')
node_cty_map.columns = (
    node_cty_map.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
node_cty_map.head()

,auto_key,county_id,county_name,nodekey,nodename,iso_code,latitude,longitude,state_id,state_short_name,version_no,insert_datetime
0,140083,39000130,OH_Greene,274,09AIRWAY69 KV 6641LOAD,PJM,39.78035,-84.05544,39,OH,20260430,2026-04-30 17:33:02.240
1,140084,39000130,OH_Greene,275,09AIRWAY69 KV 6642LOAD,PJM,39.78035,-84.05544,39,OH,20260430,2026-04-30 17:33:02.240
2,140085,39000130,OH_Greene,276,09AIRWAY69 KV BK-1,PJM,39.78035,-84.05544,39,OH,20260430,2026-04-30 17:33:02.240
3,140086,39000130,OH_Greene,277,09AIRWAY69 KV BK-2,PJM,39.78035,-84.05544,39,OH,20260430,2026-04-30 17:33:02.240
4,140087,39000484,OH_Auglaize,278,09AMSTRD69 KV BK-1,PJM,40.41857,-84.38017,39,OH,20260430,2026-04-30 17:33:02.240


## hourly load

In [4]:
offset = 0
today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

dtype_map = {
    "agg_pjm_ExternalNodeID": "int64",
    "pjm_ExternalNodeID": "int64",
    "agg_NodeName": "string",
    "NodeName": "string",
    "forecast_area": "string",
    "is_valid": "int8"
}

hourly_load = pd.read_csv(
    f"data/hourly_load_MW_by_distribution/hourly_load_MW_by_distribution_{today_str}.csv",
    skiprows=[1],
    dtype=dtype_map,
    parse_dates=["datetime_ending_ept", "insert_datetime"]
)

hourly_load = hourly_load.sort_values("datetime_ending_ept").reset_index(drop=True)

In [75]:
hourly_load.columns = (
    hourly_load.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

hourly_load = hourly_load.drop(columns=[
    "autokey",
    "insert_datetime"
], errors="ignore")

hourly_load["distribution_factor"] = (
    hourly_load["distribution_factor"]
    .astype(float)
    .round(8)
)

hourly_load = hourly_load.dropna(subset=[
    "datetime_ending_ept",
    "nodename",
    "distributed_load_mw"
])

hourly_load = hourly_load.drop_duplicates(
    subset=["datetime_ending_ept", "nodename"]
)

In [76]:
# Get all nodenames that belong to agg_nodename == "DOM" in hourly_load
dom_nodes = (
    hourly_load.loc[hourly_load["agg_nodename"] == "DOM", "nodename"]
    .dropna()
    .unique()
)

# Map those DOM nodenames to county_name using node_cty_map
dom_node_counties = (
    node_cty_map.loc[node_cty_map["nodename"].isin(dom_nodes), ["nodename", "county_name"]]
    .dropna(subset=["county_name"])
    .drop_duplicates()
)

# Get all county_name touched by those DOM nodenames
dom_counties = dom_node_counties["county_name"].unique()

# From those counties, get all nodenames in node_cty_map
all_nodes_in_dom_counties = (
    node_cty_map.loc[node_cty_map["county_name"].isin(dom_counties), "nodename"]
    .dropna()
    .unique()
)

# Select all rows from hourly_load whose nodename is in those nodenames
hourly_load_dom_counties = hourly_load.loc[
    hourly_load["nodename"].isin(all_nodes_in_dom_counties)
].copy()

In [77]:
hourly_load_dom_counties.head()

,datetime_ending_ept,agg_pjm_externalnodeid,agg_nodename,pjm_externalnodeid,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,is_valid
1048,2026-03-01,34964545,DOM,34885843,FTEUSTIS115 KV TX2,0.00018,DOMINION,14043.0,2.55583,1
1594,2026-03-01,8394954,APS,5021204,FEAGANSM138 KV T1,0.00197,AP,5316.0,10.49910,1
1595,2026-03-01,8394954,APS,31020625,FEAGANSM138 KV T2,0.00218,AP,5316.0,11.56762,1
1671,2026-03-01,8394954,APS,112585981,WILLIAM 69 KV LOAD1,0.00386,AP,5316.0,20.51444,1
1705,2026-03-01,8394954,APS,38367921,GORDONSV115 KV PRAT_AP,0.00508,AP,5316.0,27.02654,1


In [78]:
hourly_load_county = hourly_load_dom_counties[["datetime_ending_ept", "agg_nodename", "nodename", "distribution_factor", "forecast_area", "forecast_load_mw", "distributed_load_mw"]].merge(node_cty_map[["nodename", "county_name", "latitude", "longitude", "state_short_name"]], on="nodename", how="left")
hourly_load_county.head()

,datetime_ending_ept,agg_nodename,nodename,distribution_factor,forecast_area,forecast_load_mw,distributed_load_mw,county_name,latitude,longitude,state_short_name
0,2026-03-01,DOM,FTEUSTIS115 KV TX2,0.00018,DOMINION,14043.0,2.55583,VA_Newport News,37.16567,-76.58447,VA
1,2026-03-01,APS,FEAGANSM138 KV T1,0.00197,AP,5316.0,10.49910,WV_Jefferson,39.23027,-77.96287,WV
2,2026-03-01,APS,FEAGANSM138 KV T2,0.00218,AP,5316.0,11.56762,WV_Jefferson,39.23027,-77.96287,WV
3,2026-03-01,APS,WILLIAM 69 KV LOAD1,0.00386,AP,5316.0,20.51444,WV_Grant,39.19662,-79.48302,WV
4,2026-03-01,APS,GORDONSV115 KV PRAT_AP,0.00508,AP,5316.0,27.02654,VA_Orange,38.15374,-78.21017,VA


In [79]:
county_load = (
    hourly_load_county
    .groupby(
        ["datetime_ending_ept", "county_name", "state_short_name"],
        as_index=False
    )
    .agg(
        # forecast_load_mw=("forecast_load_mw", "first"),
        # distribution_factor=("distribution_factor", "sum"),
        distributed_load_mw=("distributed_load_mw", "sum"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .sort_values(["datetime_ending_ept", "county_name"])
)

county_load.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude
0,2026-03-01,DC_District of Columbia,DC,753.11005,38.890605,-77.005020
1,2026-03-01,MD_Charles,MD,214.00618,38.506417,-76.922517
2,2026-03-01,MD_Montgomery,MD,1697.80643,39.067024,-77.204228
3,2026-03-01,NC_Beaufort,NC,16.27583,35.596743,-76.875947
4,2026-03-01,NC_Bertie,NC,31.41418,36.013796,-76.955364


In [80]:
county_geo = (
    county_load
    .groupby("county_name", as_index=False)
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
)

# round to 2 decimals
county_geo["latitude"] = county_geo["latitude"].round(2)
county_geo["longitude"] = county_geo["longitude"].round(2)

county_geo.head()

,county_name,latitude,longitude
0,DC_District of Columbia,38.89,-77.01
1,MD_Charles,38.50,-76.88
2,MD_Montgomery,39.07,-77.21
3,NC_Beaufort,35.60,-76.88
4,NC_Bertie,36.01,-76.96


In [81]:
county_load = county_load[["datetime_ending_ept", "county_name", "state_short_name", "distributed_load_mw"]].merge(county_geo, on="county_name", how="left").copy()
county_load.head()

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude
0,2026-03-01,DC_District of Columbia,DC,753.11005,38.89,-77.01
1,2026-03-01,MD_Charles,MD,214.00618,38.50,-76.88
2,2026-03-01,MD_Montgomery,MD,1697.80643,39.07,-77.21
3,2026-03-01,NC_Beaufort,NC,16.27583,35.60,-76.88
4,2026-03-01,NC_Bertie,NC,31.41418,36.01,-76.96


## weather

In [82]:
county_load_weather = county_load.copy()

## classification

In [83]:
# Define the mapping dictionary
county_to_class = {
    # Solar-Driven
    'NC_Bertie': 'Solar', 'NC_Chowan': 'Solar', 'NC_Currituck': 'Solar', 'NC_Edgecombe': 'Solar', 
    'NC_Halifax': 'Solar', 'NC_Hertford': 'Solar', 'NC_Martin': 'Solar', 'NC_Nash': 'Solar', 
    'NC_Northampton': 'Solar', 'NC_Warren': 'Solar', 'NC_Washington': 'Solar', 'NC_Gates': 'Solar',
    'VA_Amelia': 'Solar', 'VA_Bedford': 'Solar', 'VA_Buckingham': 'Solar', 'VA_Essex': 'Solar', 
    'VA_Fluvanna': 'Solar', 'VA_Greensville': 'Solar', 'VA_Halifax': 'Solar', 'VA_Lancaster': 'Solar', 
    'VA_Louisa': 'Solar', 'VA_Middlesex': 'Solar', 'VA_Powhatan': 'Solar', 'VA_Prince Edward': 'Solar', 
    'VA_Richmond': 'Solar', 'VA_Shenandoah': 'Solar', 'VA_Westmoreland': 'Solar', 'VA_Brunswick': 'Solar', 
    'VA_Mecklenburg': 'Solar', 'VA_Dinwiddie': 'Solar', 'VA_King William': 'Solar', 

    # Wind-Driven
    'WV_Grant': 'Wind', 'VA_Botetourt': 'Wind', 'NC_Pasquotank': 'Wind', 'NC_Perquimans': 'Wind', 
    'VA_Virginia Beach': 'Wind',

    # Temp-Driven (Residential)
    'MD_Charles': 'Res', 'NC_Beaufort': 'Res', 'NC_Dare': 'Res', 'NC_Tyrrell': 'Res', 
    'VA_Albemarle': 'Res', 'VA_Alleghany': 'Res', 'VA_Appomattox': 'Res', 'VA_Caroline': 'Res', 
    'VA_Charlotte': 'Res',  'VA_Chesapeake': "Res", 'VA_Chesterfield': 'Res', 'VA_Covington': 'Res', 
    'VA_Cumberland': 'Res', 'VA_Emporia': 'Res', 'VA_Fauquier': 'Res', 'VA_Gloucester': 'Res', 
    'VA_Goochland': 'Res', 'VA_Hanover': 'Res', 'VA_Isle of Wight': 'Res', 'VA_James City': 'Res', 
    'VA_King George': 'Res', 'VA_Lunenburg': 'Res', 'VA_Madison': 'Res', 'VA_New Kent': 'Res', 'VA_Orange': 'Res', 
    'VA_Petersburg': 'Res', 'VA_Pittsylvania': 'Res', 'VA_Prince George': 'Res', 'VA_Rappahannock': 'Res', 
    'VA_Rockbridge': 'Res', 'VA_Rockingham': 'Res', 'VA_Southampton': 'Res', 'VA_Spotsylvania': 'Res', 
    'VA_Stafford': 'Res', 'VA_Suffolk': 'Res', 'VA_Sussex': 'Res', 'WV_Jefferson': 'Res',

    # Temp-Driven (Commercial/Industrial)
    'MD_Montgomery': 'Com/Ind', 'VA_Alexandria': 'Com/Ind', 'VA_Arlington': 'Com/Ind', 'VA_Augusta': 'Com/Ind', 
    'VA_Campbell': 'Com/Ind', 'VA_Charles City': 'Com/Ind', 'VA_Charlottesville': 'Com/Ind', 'VA_Culpeper': 'Com/Ind', 
    'VA_Falls Church': 'Com/Ind', 'VA_Franklin': 'Com/Ind', 'VA_Fredericksburg': 'Com/Ind', 'VA_Hampton': 'Com/Ind', 
    'VA_Harrisonburg': 'Com/Ind', 'VA_Hopewell': 'Com/Ind', 'VA_Loudoun': 'Com/Ind', 'VA_Manassas': 'Com/Ind', 
    'VA_Manassas Park': 'Com/Ind', 'VA_Newport News': 'Com/Ind', 'VA_Norfolk': 'Com/Ind', 'VA_Nottoway': 'Com/Ind', 
    'VA_Poquoson': 'Com/Ind', 'VA_Portsmouth': 'Com/Ind', 'VA_Prince William': 'Com/Ind', 'VA_Surry': 'Com/Ind', 
    'VA_Waynesboro': 'Com/Ind', 'VA_York': 'Com/Ind', 'VA_Henrico': 'Com/Ind', 'VA_Fairfax': 'Com/Ind', 
    'DC_District of Columbia': 'Com/Ind'
    # Unclear-Driven
}

# Apply to your dataframe
# Assuming 'county_name' is the column with names like 'VA_Fairfax'
county_load_weather['classification'] = county_load_weather['county_name'].map(county_to_class)

In [ ]:
code_to_label = {
    'S': 'Solar',
    'W': 'Wind',
    'R': 'Res',
    'C': 'Com/Ind',
    'U': 'Unclear'
}

county_to_class_name = {
    'DC_District of Columbia': 'C', 
    
    'MD_Charles': 'R', 'MD_Montgomery': 'C', 
    
    'NC_Beaufort': 'S', 'NC_Bertie': 'S', 'NC_Chowan': 'U', 'NC_Currituck': 'S', 
    'NC_Dare': 'R', 'NC_Edgecombe': 'S', 'NC_Gates': 'U', 'NC_Halifax': 'S', 
    'NC_Hertford': 'S', 'NC_Martin': 'S', 'NC_Nash': 'U', 'NC_Northampton': 'S', 
    'NC_Pasquotank': 'U', 'NC_Perquimans': 'S', 'NC_Tyrrell': 'S', 'NC_Warren': 'S', 
    'NC_Washington': 'U',

    'VA_Albemarle': 'R', 'VA_Alexandria': 'C', 'VA_Alleghany': 'U', 'VA_Amelia': 'U',
    'VA_Appomattox': 'R', 'VA_Arlington': 'C', 'VA_Augusta': 'R', 'VA_Bedford': 'R',
    'VA_Botetourt': 'U', 'VA_Brunswick': 'U', 'VA_Buckingham': 'S', 'VA_Campbell': 'U',
    'VA_Caroline': 'R', 'VA_Charles City': 'C', 'VA_Charlotte': 'U', 'VA_Charlottesville': 'C',
    'VA_Chesapeake': 'R', 'VA_Chesterfield': 'R', 'VA_Covington': 'U', 'VA_Culpeper': 'C',
    'VA_Cumberland': 'R', 'VA_Dinwiddie': 'U', 'VA_Emporia': 'C', 'VA_Essex': 'S',
    'VA_Fairfax': 'C', 'VA_Falls Church': 'C', 'VA_Fauquier': 'R', 'VA_Fluvanna': 'S',
    'VA_Franklin': 'C', 'VA_Fredericksburg': 'C', 'VA_Gloucester': 'C', 'VA_Goochland': 'R',
    'VA_Greensville': 'U', 'VA_Halifax': 'R', 'VA_Hampton': 'C', 'VA_Hanover': 'R',
    'VA_Harrisonburg': 'C', 'VA_Henrico': 'C', 'VA_Hopewell': 'C', 'VA_Isle of Wight': 'R',
    'VA_James City': 'R', 'VA_King George': 'U', 'VA_King William': 'U', 'VA_Lancaster': 'S',
    'VA_Loudoun': 'C', 'VA_Louisa': 'S', 'VA_Lunenburg': 'R', 'VA_Madison': 'U',
    'VA_Manassas': 'C', 'VA_Manassas Park': 'C', 'VA_Mecklenburg': 'C', 'VA_Middlesex': 'R',
    'VA_New Kent': 'R', 'VA_Newport News': 'C', 'VA_Norfolk': 'C', 'VA_Nottoway': 'C',
    'VA_Orange': 'R', 'VA_Petersburg': 'R', 'VA_Pittsylvania': 'R', 'VA_Poquoson': 'C',
    'VA_Portsmouth': 'C', 'VA_Powhatan': 'S', 'VA_Prince Edward': 'S', 'VA_Prince George': 'R',
    'VA_Prince William': 'C', 'VA_Rappahannock': 'U', 'VA_Richmond': 'C', 'VA_Rockbridge': 'R',
    'VA_Rockingham': 'R', 'VA_Shenandoah': 'R', 'VA_Southampton': 'U', 'VA_Spotsylvania': 'R',
    'VA_Stafford': 'R', 'VA_Suffolk': 'R', 'VA_Surry': 'C', 'VA_Sussex': 'C',
    'VA_Virginia Beach': 'R', 'VA_Waynesboro': 'C', 'VA_Westmoreland': 'R', 'VA_York': 'C',

    'WV_Grant': 'W', 'WV_Jefferson': 'R'
    # worth notice: Prince George
}

county_load_weather['class_code'] = county_load_weather['county_name'].map(county_to_class_name)
county_load_weather['class_name'] = county_load_weather['class_code'].map(code_to_label)

In [ ]:
# --- Prepare data ---
df = county_load_weather.copy()
df["datetime_ending_ept"] = pd.to_datetime(df["datetime_ending_ept"])

df = df[
    df["datetime_ending_ept"] >= (pd.Timestamp.now() - pd.Timedelta(days=7))
].copy()

df["date"] = df["datetime_ending_ept"].dt.date
df["hour"] = df["datetime_ending_ept"].dt.hour

# --- Dropdown widget ---
county_list = sorted(df["county_name"].unique())

dropdown = widgets.Dropdown(
    options=county_list,
    description="County:",
    layout=widgets.Layout(width='300px')
)

# --- Plot function ---
def plot_county(county):
    sub = df[df["county_name"] == county]

    fig = px.line(
        sub,
        x="hour",
        y="distributed_load_mw",
        color="date",
        title=f"Hourly Load Profile — {county}",
        height=500,   # adjust here
        width=900     # adjust here
    )
    fig.show()

# --- Link dropdown to plot ---
output = widgets.interactive_output(plot_county, {"county": dropdown})

# --- Display ---
display(dropdown, output)

Dropdown(description='County:', layout=Layout(width='300px'), options=('DC_District of Columbia', 'MD_Charles'…

Output()

In [93]:
county_load_weather

,datetime_ending_ept,county_name,state_short_name,distributed_load_mw,latitude,longitude,classification,class_code,class_name
0,2026-03-01,DC_District of Columbia,DC,753.11005,38.89,-77.01,Com/Ind,C,Com/Ind
1,2026-03-01,MD_Charles,MD,214.00618,38.50,-76.88,Res,R,Res
2,2026-03-01,MD_Montgomery,MD,1697.80643,39.07,-77.21,Com/Ind,C,Com/Ind
3,2026-03-01,NC_Beaufort,NC,16.27583,35.60,-76.88,Res,R,Res
4,2026-03-01,NC_Bertie,NC,31.41418,36.01,-76.96,Solar,S,Solar
...,...,...,...,...,...,...,...,...,...
112303,2026-04-16,VA_Waynesboro,VA,20.48634,38.06,-78.89,Com/Ind,C,Com/Ind
112304,2026-04-16,VA_Westmoreland,VA,31.36361,38.09,-76.89,Solar,S,Solar
112305,2026-04-16,VA_York,VA,290.82472,37.19,-76.48,Com/Ind,C,Com/Ind
112306,2026-04-16,WV_Grant,WV,12.46833,39.20,-79.44,Wind,W,Wind


In [87]:
county_load_weather.groupby("classification")["county_name"].nunique()

classification
Com/Ind    29
Res        37
Solar      31
Wind        5
Name: county_name, dtype: int64

In [94]:
# 1. Load and Prepare the Data

# df = pd.read_csv('your_data.csv') 
df = county_load_weather.copy() 

# Convert your datetime column to actual datetime objects
df['datetime'] = pd.to_datetime(df['datetime_ending_ept'])

# Extract 'date' and 'hour' for the line chart
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# Load the shapefile
shapefile_path = 'data/cb_2023_us_county_500k/cb_2023_us_county_500k.shp'
gdf = gpd.read_file(shapefile_path)

# Convert shapefile to standard latitude/longitude format (WGS84)
gdf = gdf.to_crs(epsg=4326)

# 2. Calculate Yesterday's Peak Load

# Find the most recent date in the dataset, then subtract 1 day to get 'yesterday'
latest_date_overall = df['date'].max()
yesterday_date = latest_date_overall - pd.Timedelta(days=2)

# Filter for yesterday, group by county, and find the maximum load
yesterday_data = df[df['date'] == yesterday_date]
yesterday_peaks = yesterday_data.groupby('county_name')['distributed_load_mw'].max().reset_index()
yesterday_peaks.rename(columns={'distributed_load_mw': 'yesterday_peak_mw'}, inplace=True)

# 3. Spatial Join & Base Map Creation

# We only need one row per county to color the map
df_map = df.drop_duplicates(subset=['county_name']).copy()

# Merge the peak loads into our map dataframe
df_map = df_map.merge(yesterday_peaks, on='county_name', how='left')

# 1. Extract the clean county name (e.g., "VA_Rockingham" -> "Rockingham")
df_map['county_clean'] = df_map['county_name'].apply(lambda x: x.split('_', 1)[-1] if '_' in x else x)

# 2. Create a dictionary to map your state abbreviations to Census STATEFP codes.
state_fips_map = {
    'VA': '51', 'WV': '54', 'NC': '37', 'MD': '24', 'DC': '11', 'PA': '42'
}
df_map['STATEFP'] = df_map['state_short_name'].map(state_fips_map)

# 3. Merge the GeoDataFrame and Pandas DataFrame
df_joined = gdf.merge(
    df_map, 
    left_on=['STATEFP', 'NAME'], 
    right_on=['STATEFP', 'county_clean'], 
    how='inner'
)

# Create the choropleth map
fig_map = px.choropleth(
    df_joined,
    geojson=gdf.__geo_interface__,
    locations='GEOID',                 
    featureidkey="properties.GEOID",   
    color='class_name',
    color_discrete_map={
        'Com/Ind': '#0072B2',
        'Solar': '#E69F00',
        'Wind': '#009E73',
        'Res': '#CC79A7',
        'Unclear': '#999999'
    },
    # Use a dictionary for hover_data to control visibility and order
    hover_data={
        'county_name': True,         # This stays at customdata[0] for your callback
        'class_name': True,
        'yesterday_peak_mw': ':.2f', # Show peak load formatted to 2 decimal places
        'GEOID': False,              # Hide the raw GEOID
        'STATEFP': False             # Hide state FIPS code if it pops up
    },
    # Rename variables so they look clean on the hover tooltip
    labels={
        'county_name': 'County',
        'class_name': 'Class_name',
        'yesterday_peak_mw': "Yesterday's Peak Load (MW)"
    }
)

fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(
    margin={"r":0,"t":0,"l":0,"b":0},
    clickmode='event+select' 
)

# 4. Build the Dash App

app = Dash(__name__)

app.layout = html.Div([
    html.H2("County Classification and Load Distribution", style={'textAlign': 'center'}),
    
    html.Div([
        html.Div([
            dcc.Graph(
                id='county-map', 
                figure=fig_map, 
                style={'height': '70vh'}
            )
        ], style={'width': '60%', 'display': 'inline-block', 'verticalAlign': 'top'}),
        
        html.Div([
            html.Div(id='click-output-container', style={'padding': '20px'})
        ], style={'width': '40%', 'display': 'inline-block', 'verticalAlign': 'top'})
    ])
])

@app.callback(
    Output('click-output-container', 'children'),
    Input('county-map', 'clickData')
)
def display_click_data(clickData):
    if clickData is None:
        return html.H4("Click on a county on the map to see its 7-day load profile.", 
                       style={'color': 'gray', 'textAlign': 'center', 'marginTop': '100px'})

    # Extract the county name. Because 'county_name' is the first True item in 
    # our hover_data dictionary, it safely remains at customdata[0].
    pt = clickData['points'][0]
    clicked_county = pt['customdata'][0] 

    df_county = df[df['county_name'] == clicked_county].copy()
    
    if not df_county.empty:
        latest_date = df_county['date'].max()
        past_7_days = [latest_date - pd.Timedelta(days=i) for i in range(7)]
        df_county_7d = df_county[df_county['date'].isin(past_7_days)]

        fig_line = px.line(
            df_county_7d,
            x='hour',
            y='distributed_load_mw',
            color='date', 
            title=f"7-Day Load Profile: {clicked_county}",
            labels={'hour': 'Hour', 'distributed_load_mw': 'Load (MW)', 'date': 'Date'}
        )
        
        fig_line.update_layout(
            margin=dict(l=40, r=20, t=50, b=40),
            height=400,
            xaxis=dict(tickmode='linear', dtick=4) 
        )

        return dcc.Graph(figure=fig_line)
    
    return html.P("No time-series data found for this county.")

if __name__ == '__main__':
    app.run(debug=True)